# Item 15b — os 12 treinos do encoder de atenção

Segundo notebook da Phase 7 ([spec](../.specs/features/phase-7-atencao/spec.md), ATT-03 e
ATT-04). Treina, com a configuração que o item 15a escolheu e **sem mudá-la**:

| variante | grafo que enxerga | papel ([ADR-0006](../docs/adr/0006-precompromisso-fase-atencao.md)) |
|---|---|---|
| `atencao` (`A`) | o grafo real | a variante |
| `atencao_emb` (`A-emb`) | `rewire_preserving_degree(g, seed=seed)`, `edge_attr` alinhado à aresta original | isola a topologia sob o encoder novo |

× 2 splits (`current`, `pre_pandemia`) × 3 seeds (42, 43, 44) = **12 treinos**, com
`max_cotraj_edges = 30_000` e 35 épocas, o protocolo da escada.

**Runtime: GPU (T4).** O notebook aborta logo depois do setup se não houver GPU.

**Portões.** `config_escolhida.json` precisa existir e o hash da configuração precisa bater; um
checkpoint gravado com outro hash também aborta. Cada treino passa por `check_param_budget`
(453.121 ± 10%) antes de começar.

**Perda NaN.** Se um treino diverge, a falha é registrada, o treino é repetido **uma vez** com a
mesma configuração e outra inicialização, e as duas tentativas ficam gravadas (edge case da spec).

**Retomada.** Cada treino grava seu checkpoint no Drive assim que termina e o laço pula o que já
existe. `TEMPO_LIMITE_MIN` corta o laço antes da desconexão.

## 0. Ambiente — Colab (GPU) ou local

No Colab, clona o repositório (código + artefatos de dados versionados) e instala o PyG.
Ative a GPU em *Ambiente de execução → Alterar tipo de runtime → GPU*.
Repo privado: cole um PAT em `GITHUB_TOKEN`.

> Os grafos `hetero_full_current.pt` e `hetero_full_pre_pandemia.pt` precisam estar **commitados**
> antes de clonar: os CSVs brutos da rede de gêneros não são versionados, então o grafo não pode
> ser reconstruído aqui dentro.

In [ ]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

DATA_FILES = [
    "data/processed/graph/hetero_full_current.pt",
    "data/processed/graph/hetero_full_pre_pandemia.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if IS_COLAB:
    if Path(REPO_DIR, "pyproject.toml").exists():
        # O runtime sobrevive ao restart do kernel: um clone de sessão anterior ficaria
        # para trás em silêncio e o import viria do código velho. Sincroniza sempre.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    else:
        tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
        print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
        if _clone(url).returncode != 0:
            from getpass import getpass
            tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
            _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
    os.chdir(REPO_DIR)
    print("commit em uso:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "pyarrow"], check=True)
    faltando = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
    print("✓ Colab pronto. cwd =", os.getcwd())
    if faltando:
        print("⚠️ FALTAM no repositório:", faltando)
else:
    print("Local — nada a clonar.")

In [ ]:
# Portão de GPU: este notebook treina/avalia modelos e só roda numa GPU do Colab (T4).
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Sem GPU: este notebook precisa de um runtime com GPU (T4). No Colab: Ambiente de "
        "execução → Alterar tipo de runtime → GPU, e rode de novo. Abortado antes de qualquer "
        "treino ou avaliação.")
DEVICE = "cuda"
print("DEVICE =", DEVICE, "|", torch.cuda.get_device_name(0))

In [ ]:
import json, time, hashlib, math, dataclasses, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

_anchor = Path(globals().get("__vsc_ipynb_file__", os.getcwd()))
if not _anchor.exists():
    _anchor = Path(os.getcwd())
ROOT = _anchor if _anchor.is_dir() else _anchor.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"raiz do projeto não encontrada a partir de {_anchor}"
sys.path.insert(0, str(ROOT / "src"))

%matplotlib inline
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from music_diffusion_gnn.evaluation.stats import aggregate_seeds
from music_diffusion_gnn.evaluation.veredito_atencao import SAGE_N_PARAMS, budget_block
from music_diffusion_gnn.graph.build import graph_path
from music_diffusion_gnn.graph.controls import rewire_preserving_degree
from music_diffusion_gnn.models.diffusion_gnn import MusicDiffusionGNN
from music_diffusion_gnn.training.dataset import (
    aggregate_weekly, build_pop_bank, build_samples, temporal_split,
)
from music_diffusion_gnn.training.trainer import (
    Config, check_param_budget, checkpoint_dict, train_one,
)

GRAPH_DIR = ROOT / "data" / "processed" / "graph"
NMAP_PATH = GRAPH_DIR / "node_id_map.json"
TS_PATH   = ROOT / "data" / "processed" / "timeseries.parquet"
print("ROOT =", ROOT, "| DEVICE =", DEVICE)

In [ ]:
# Artefatos DURÁVEIS no Drive (sobrevivem à desconexão do Colab)
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/music-influence-gnn/atencao")
else:
    BASE = ROOT / "results" / "atencao"
CONFIG_JSON = BASE / "config_escolhida.json"
OUT = BASE / "treinos"
OUT.mkdir(parents=True, exist_ok=True)
print("configuração ←", CONFIG_JSON)
print("artefatos    →", OUT)

## 1. A configuração escolhida no ajuste

Lida de `config_escolhida.json` (item 15a). O hash é recalculado a partir dos campos gravados: se
o arquivo faltar ou o hash não bater, o notebook para. Nenhuma constante de configuração é
digitada aqui; só a seed muda entre os treinos.

In [ ]:
if not CONFIG_JSON.exists():
    raise RuntimeError(f"{CONFIG_JSON} não existe: rode item15a_ajuste_atencao_colab.ipynb até o "
                       "fim antes de treinar. A configuração dos 12 treinos sai do ajuste (ADR-0006).")
registro = json.loads(CONFIG_JSON.read_text())
CAMPOS = registro["config"]
HASH_CFG = hashlib.sha256(json.dumps(CAMPOS, sort_keys=True).encode()).hexdigest()
if HASH_CFG != registro["hash"]:
    raise RuntimeError(
        f"a configuração em {CONFIG_JSON.name} mudou depois de criada (hash gravado "
        f"{registro['hash'][:12]}, recalculado {HASH_CFG[:12]}). Ela não pode mudar depois do "
        "ajuste (ATT-03, AC2): restaure o arquivo original.")
assert CAMPOS["encoder"] == "gatv2", CAMPOS["encoder"]
assert CAMPOS["max_cotraj_edges"] == 30_000, "o protocolo fixa max_cotraj_edges = 30_000 (AC6)"
assert CAMPOS["max_epochs"] == 35, "o protocolo fixa 35 épocas (AC6)"
print(f"configuração {registro['config_str']} (hash {HASH_CFG[:12]}), escolhida em "
      f"{registro['criado_em']}; empate decidiu: {registro['selecao']['regra_de_empate_decidiu']}")
display(pd.Series(CAMPOS, name="valor").to_frame())

## 2. O plano de treinos

A ordem é por regime e por seed, com as duas variantes juntas: assim que a seed 42 de `current`
fecha, já existe um par `A`/`A-emb` para olhar, mesmo que a sessão caia depois. A seed do
religamento do `A-emb` é a seed do experimento, como no item 08: a dispersão entre seeds do
controle inclui a do próprio embaralhamento.

In [ ]:
SEEDS     = [42, 43, 44]
REGIMES   = ["current", "pre_pandemia"]
VARIANTES = ["atencao", "atencao_emb"]
TEMPO_LIMITE_MIN = 150                     # corta o laço antes da sessão do Colab cair
SEMENTE_REPETICAO = 1000                   # 2ª tentativa após NaN: seed de treino = seed + 1000

PLANO = [(v, r, s) for r in REGIMES for s in SEEDS for v in VARIANTES]
assert len(PLANO) == 12

def variante_grafo(g, variante: str, seed: int):
    """O grafo que cada variante enxerga. Nenhuma das funções muta `g`."""
    if variante == "atencao":
        return g
    if variante == "atencao_emb":
        return rewire_preserving_degree(g, seed=seed)
    raise ValueError(variante)

def graph_seed_de(variante: str, seed: int):
    return seed if variante == "atencao_emb" else None

for i, p in enumerate(PLANO, 1):
    print(f"  {i:2d}. {p[0]:<12s} {p[1]:<13s} seed {p[2]}")

## 3. Dados por regime

Como no item 06: amostras construídas uma vez por regime, `first_seen` sobre a série completa.
A versão do grafo (nome e hash do arquivo) vai para cada checkpoint.

In [ ]:
ts = pd.read_parquet(TS_PATH)
weekly = aggregate_weekly(ts)
first_seen = weekly.groupby(["song_id", "chart"], observed=True)["week"].min().to_dict()

_cache_dados: dict[str, dict] = {}

def dados(regime: str) -> dict:
    if regime not in _cache_dados:
        p = graph_path(regime, GRAPH_DIR)
        g = torch.load(p, weights_only=False)
        splits_df = temporal_split(weekly, regime=regime)
        samples = {
            nome: build_samples(splits_df[nome], W=CAMPOS["W"], node_id_map_path=NMAP_PATH,
                                first_seen=first_seen)
            for nome in ("train", "val")
        }
        versao = f"{p.name}@{hashlib.sha256(p.read_bytes()).hexdigest()[:12]}"
        print(f"{regime}: " + " ".join(f"{k}={len(v):,}" for k, v in samples.items())
              + f" | grafo {versao}")
        _cache_dados[regime] = {
            "g": g, "samples": samples, "graph_version": versao,
            "pop_bank": build_pop_bank(weekly, NMAP_PATH, n_music=g["music"].num_nodes),
        }
    return _cache_dados[regime]

_ = dados("current")

## 4. Orçamento de parâmetros contra a proposta SAGE

O `GATv2Conv` tem parâmetros lazy: o forward de uma semana materializa a contagem. As duas
variantes instanciam a mesma classe com a mesma configuração; o grafo só muda quais mensagens
trafegam. A contagem é comparada com os 453.121 parâmetros da proposta SAGE, e o mesmo
`check_param_budget` roda de novo antes de cada treino na seção 5.

In [ ]:
def cfg_de(seed: int) -> Config:
    return Config(**CAMPOS, seed=seed)

def parametros_aquecidos(cfg: Config, g_var, pop_bank) -> int:
    """Conta os parâmetros do modelo que train_one vai construir, depois do forward de uma semana."""
    m = MusicDiffusionGNN(g_var.metadata(), hidden=cfg.hidden, layers=cfg.layers,
                          dropout=cfg.dropout, pop_bank=pop_bank, encoder=cfg.encoder,
                          heads=cfg.heads, attn_dropout=cfg.attn_dropout).to(DEVICE)
    with torch.no_grad():
        m.encode_weeks(g_var.to(DEVICE), [cfg.W], max_cotraj_edges=cfg.max_cotraj_edges)
    n = m.count_params()
    del m
    torch.cuda.empty_cache()
    return n

d = dados("current")
orc = pd.DataFrame([
    {"variante": v, "n_params": parametros_aquecidos(cfg_de(42), variante_grafo(d["g"], v, 42),
                                                     d["pop_bank"])}
    for v in VARIANTES
])
orc["proposta_sage"] = SAGE_N_PARAMS
orc["diferenca"] = orc.n_params / SAGE_N_PARAMS - 1
display(orc)
assert orc.n_params.nunique() == 1, "as duas variantes deixaram de ter o mesmo orçamento"
for n in orc.n_params:
    check_param_budget(int(n))
print(budget_block(int(orc.n_params.iloc[0])))

## 5. O laço de treino

Para cada treino: aquecimento e `check_param_budget` (fora da faixa, a exceção interrompe a célula
e o treino não começa), `train_one`, checkpoint com `checkpoint_dict` (regime, seed, variante,
configuração, versão do grafo, `graph_seed` do embaralhado) mais o hash da configuração e o
registro das tentativas.

**Falha (perda NaN).** Uma tentativa falha quando `train_one` lança exceção (com perda NaN desde a
primeira época não há melhor estado para recarregar) ou quando alguma perda de treino ou validação
não é finita. A falha é gravada em `falha_{...}_t{n}.json` e o treino é repetido uma vez, com a
seed de treino `seed + 1000` (outra inicialização; a ordem dos lotes também muda) e o mesmo grafo
(`graph_seed` continua sendo a seed do experimento). Falta de memória da GPU não é tratada como
NaN: ela interrompe a célula. Se as duas tentativas falharem, o treino fica como lacuna declarada e
o laço segue; uma sessão nova não tenta uma terceira vez.

Rode esta célula quantas vezes precisar. Ela sempre continua de onde parou.

In [ ]:
def nome(variante, regime, seed):
    return f"gnn_{variante}_{regime}_seed{seed}"

def falha_path(variante, regime, seed, tentativa):
    return OUT / f"falha_{nome(variante, regime, seed)}_t{tentativa}.json"

def tentar(cfg: Config, d: dict, g_var):
    """Uma tentativa de treino. Devolve (TrainResult ou None, registro da tentativa)."""
    reg = {"seed_treino": cfg.seed, "inicio": time.strftime("%Y-%m-%d %H:%M:%S")}
    try:
        r = train_one(cfg, {"train": d["samples"]["train"], "val": d["samples"]["val"]},
                      g_var, device=DEVICE, pop_bank=d["pop_bank"])
    except torch.cuda.OutOfMemoryError:
        raise
    except Exception as e:  # perda NaN desde a 1ª época: train_one não tem estado para recarregar
        reg |= {"ok": False, "erro": f"{type(e).__name__}: {e}",
                "traceback": traceback.format_exc()[-2000:]}
        return None, reg
    curvas = list(r.train_curve) + list(r.val_curve) + [r.val_mse]
    finita = all(math.isfinite(x) for x in curvas)
    reg |= {"ok": finita, "val_mse": r.val_mse, "train_curve": r.train_curve,
            "val_curve": r.val_curve, "elapsed_sec": r.elapsed_sec}
    if not finita:
        reg["erro"] = "perda não finita (NaN/inf) numa época"
    return (r if finita else None), reg

t_inicio = time.time()
for variante, regime, seed in PLANO:
    ckpt = OUT / f"{nome(variante, regime, seed)}.pt"
    if ckpt.exists():
        r = torch.load(ckpt, map_location="cpu", weights_only=False)
        if r.get("config_hash") != HASH_CFG:
            raise RuntimeError(f"{ckpt.name} foi treinado com outra configuração "
                               f"(hash {str(r.get('config_hash'))[:12]}, esperado {HASH_CFG[:12]})")
        print(f"{nome(variante, regime, seed)}: pronto (val_mse={r['val_mse']:.6f}) — pulando")
        continue
    if falha_path(variante, regime, seed, 1).exists() and falha_path(variante, regime, seed, 2).exists():
        print(f"{nome(variante, regime, seed)}: as duas tentativas falharam — lacuna declarada, pulando")
        continue

    decorrido = (time.time() - t_inicio) / 60
    if decorrido > TEMPO_LIMITE_MIN:
        print(f"\nlimite de {TEMPO_LIMITE_MIN} min atingido ({decorrido:.0f} min). Rode a célula de "
              "novo numa sessão nova: os checkpoints prontos serão pulados.")
        break

    d = dados(regime)
    g_var = variante_grafo(d["g"], variante, seed)
    tentativas = []
    if falha_path(variante, regime, seed, 1).exists():   # a 1ª falhou numa sessão anterior
        tentativas.append(json.loads(falha_path(variante, regime, seed, 1).read_text()))
    resultado = None
    while resultado is None and len(tentativas) < 2:
        n_tent = len(tentativas) + 1
        cfg = cfg_de(seed if n_tent == 1 else seed + SEMENTE_REPETICAO)
        n_aquecido = parametros_aquecidos(cfg, g_var, d["pop_bank"])
        check_param_budget(n_aquecido)                   # aborta fora de 453.121 ± 10% (AC6)
        print(f"\n▶ {nome(variante, regime, seed)} — tentativa {n_tent}, {cfg}, "
              f"{n_aquecido:,} parâmetros")
        resultado, reg = tentar(cfg, d, g_var)
        reg["tentativa"] = n_tent
        tentativas.append(reg)
        if resultado is None:
            falha_path(variante, regime, seed, n_tent).write_text(
                json.dumps(reg, indent=2, ensure_ascii=False, default=str))
            print(f"  ✗ falhou: {reg['erro']} → {falha_path(variante, regime, seed, n_tent).name}")

    if resultado is None:
        print(f"  as duas tentativas falharam: {nome(variante, regime, seed)} fica como lacuna declarada")
    else:
        assert resultado.n_params == n_aquecido, (resultado.n_params, n_aquecido)
        check_param_budget(resultado.n_params)
        ck = checkpoint_dict(cfg, resultado, variante=variante, split_regime=regime,
                             graph_seed=graph_seed_de(variante, seed),
                             graph_version=d["graph_version"])
        ck |= {"seed": seed, "seed_treino": cfg.seed, "config_hash": HASH_CFG,
               "config_escolhida": CAMPOS, "tentativas": tentativas,
               "elapsed_sec": resultado.elapsed_sec}
        torch.save(ck, ckpt)
        print(f"  val_mse={resultado.val_mse:.6f}  épocas={len(resultado.val_curve)}  "
              f"t={resultado.elapsed_sec/60:.1f}min → {ckpt.name}")
    del g_var
    torch.cuda.empty_cache()

feitos = sum((OUT / f"{nome(*p)}.pt").exists() for p in PLANO)
print(f"\nfeitos: {feitos} de {len(PLANO)}")

## 6. Onde a matriz está

`val_mse` é a métrica de parada, não a comparação: a comparação é o Modo 2 do item 15c e o
veredito do item 15d. Esta tabela mostra o que falta treinar, quantas tentativas cada treino
precisou e a contagem de parâmetros de cada checkpoint.

In [ ]:
linhas = []
for variante, regime, seed in PLANO:
    ck_path = OUT / f"{nome(variante, regime, seed)}.pt"
    falhas = [t for t in (1, 2) if falha_path(variante, regime, seed, t).exists()]
    base = {"variante": variante, "regime": regime, "seed": seed, "falhas": len(falhas)}
    if not ck_path.exists():
        linhas.append(base | {"val_mse": np.nan, "status": "lacuna" if len(falhas) == 2 else "falta"})
        continue
    r = torch.load(ck_path, map_location="cpu", weights_only=False)
    check_param_budget(r["n_params"])
    linhas.append(base | {"val_mse": r["val_mse"], "n_params": r["n_params"],
                          "tentativas": len(r["tentativas"]), "graph_seed": r["graph_seed"],
                          "graph_version": r["graph_version"], "config_str": r["config_str"],
                          "epocas": len(r["val_curve"]), "elapsed_min": r["elapsed_sec"] / 60,
                          "status": "pronto"})
matriz = pd.DataFrame(linhas)
display(matriz)
display(matriz.pivot_table(index=["regime", "seed"], columns="variante", values="val_mse"))

agg = []
for (variante, regime), grp in matriz.dropna(subset=["val_mse"]).groupby(["variante", "regime"]):
    media, desvio = aggregate_seeds(grp["val_mse"].to_numpy())
    agg.append({"variante": variante, "regime": regime, "n_seeds": len(grp),
                "val_mse_medio": media, "val_mse_desvio": desvio})
if agg:
    display(pd.DataFrame(agg).sort_values(["regime", "val_mse_medio"]))
matriz.to_parquet(OUT / "matriz_val_mse.parquet", index=False)

## 7. O que levar de volta

Do Drive (`atencao/treinos/`):

- `gnn_{variante}_{regime}_seed{seed}.pt` — os 12 checkpoints, cada um com os metadados de
  `checkpoint_dict`, o hash da configuração e o registro das tentativas;
- `falha_*_t{n}.json` — as tentativas que divergiram, se houver;
- `matriz_val_mse.parquet` — a grade de `val_mse`.

Em seguida, `item15c_avaliacao_atencao_colab.ipynb` (Modo 2).